# Download raw datasets

Downloads the two e-commerce sources listed in the README and stores them under `src/data/raw/` (gitignored):

| Source | What | Output |
|---|---|---|
| [Bitext Customer Support](https://huggingface.co/datasets/bitext/Bitext-customer-support-llm-chatbot-training-dataset) | ~27k support questions with reference answers, labelled by `category` / `intent` | `raw/bitext/bitext_customer_support.jsonl` |
| [Amazon Q/A](https://cseweb.ucsd.edu/~jmcauley/datasets.html#amazon_qa) (McAuley Lab, `qa_*.json.gz`) | Product questions with one community answer each | `raw/amazon/qa_<Category>.json.gz` |
| Amazon product metadata 2014 (`meta_*.json.gz`, SNAP Stanford mirror) | Title, brand, price, description, categories per ASIN: the context the questions need | `raw/amazon/meta_<Category>.json.gz` |

The Q/A files are joined with the metadata on `asin` and saved as `raw/amazon/amazon_qa_<Category>.jsonl`.

All outputs are JSONL (one JSON object per line, UTF-8). Missing values are written as `null`.

**Notes**
- The Q/A data was collected against the 2014 review/metadata dump, so the 2014 `meta_*` files are used (the 2018 `amazon_v2` files are much larger and cover a different item set).
- Both Amazon file types are Python dict literals per line, not strict JSON, so they are parsed with `ast.literal_eval`.
- Answers in both sources are **references only**. Grading stays rubric-based.
- Licences: Bitext is CDLA-Sharing-1.0. The Amazon data is for research use. Do not redistribute the raw files.
- Downloads are skipped if the file already exists, so the notebook can be re-run safely.

In [ ]:
import ast
import gzip
import html
import re
from pathlib import Path

import pandas as pd
import requests
from tqdm.auto import tqdm

## Configuration

Pick the Amazon categories here. The default is three small categories (~36 MB of metadata). `"ALL"` downloads all 21 categories (~1.9 GB of metadata; Electronics and Clothing alone are ~470 MB).

In [ ]:
AMAZON_CATEGORIES = ["Appliances", "Software", "Video_Games"]  # or "ALL"

ALL_AMAZON_CATEGORIES = [
    "Appliances", "Arts_Crafts_and_Sewing", "Automotive", "Baby", "Beauty",
    "Cell_Phones_and_Accessories", "Clothing_Shoes_and_Jewelry", "Electronics",
    "Grocery_and_Gourmet_Food", "Health_and_Personal_Care", "Home_and_Kitchen",
    "Industrial_and_Scientific", "Musical_Instruments", "Office_Products",
    "Patio_Lawn_and_Garden", "Pet_Supplies", "Software", "Sports_and_Outdoors",
    "Tools_and_Home_Improvement", "Toys_and_Games", "Video_Games",
]
if AMAZON_CATEGORIES == "ALL":
    AMAZON_CATEGORIES = ALL_AMAZON_CATEGORIES
unknown = set(AMAZON_CATEGORIES) - set(ALL_AMAZON_CATEGORIES)
assert not unknown, f"Unknown categories: {unknown}"

BITEXT_DATASET = "bitext/Bitext-customer-support-llm-chatbot-training-dataset"
AMAZON_QA_URL = "https://mcauleylab.ucsd.edu/public_datasets/data/amazon/qa/qa_{cat}.json.gz"
AMAZON_META_URL = "https://snap.stanford.edu/data/amazon/productGraph/categoryFiles/meta_{cat}.json.gz"

# Resolve paths from the repo root so the notebook works no matter where Jupyter was started.
REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / ".git").exists())
RAW_DIR = REPO_ROOT / "src" / "data" / "raw"
BITEXT_DIR = RAW_DIR / "bitext"
AMAZON_DIR = RAW_DIR / "amazon"
for d in (BITEXT_DIR, AMAZON_DIR):
    d.mkdir(parents=True, exist_ok=True)

print("Raw data dir:", RAW_DIR)
print("Amazon categories:", AMAZON_CATEGORIES)

## Helpers

In [ ]:
def download(url: str, dest: Path, chunk_size: int = 1 << 20) -> Path:
    """Stream a file to disk. Skips existing files; writes to a .part file first so aborted downloads are never reused."""
    if dest.exists():
        print(f"skip (exists): {dest.name}")
        return dest
    tmp = dest.with_suffix(dest.suffix + ".part")
    with requests.get(url, stream=True, timeout=60) as r:
        r.raise_for_status()
        total = int(r.headers.get("content-length", 0)) or None
        with open(tmp, "wb") as f, tqdm(total=total, unit="B", unit_scale=True, desc=dest.name) as bar:
            for chunk in r.iter_content(chunk_size):
                f.write(chunk)
                bar.update(len(chunk))
    tmp.replace(dest)
    return dest


def write_jsonl(df: pd.DataFrame, path: Path) -> None:
    df.to_json(path, orient="records", lines=True, force_ascii=False)


def read_jsonl(path: Path) -> pd.DataFrame:
    # dtype=False keeps ASINs like "0970408285" as strings instead of turning them into ints.
    return pd.read_json(path, lines=True, dtype=False, convert_dates=False)


def iter_py_literals(path: Path):
    """Yield one dict per line from a gzipped file of Python dict literals."""
    with gzip.open(path, "rt", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                yield ast.literal_eval(line)


ASIN_RE = re.compile(r"""['"]asin['"]:\s*['"]([^'"]+)['"]""")

def load_meta_for(path: Path, asins: set[str]) -> pd.DataFrame:
    """Load only the metadata rows whose ASIN appears in `asins`.
    A regex pre-filter avoids literal_eval on lines we don't need (meta files are much larger than the Q/A files)."""
    rows = []
    with gzip.open(path, "rt", encoding="utf-8") as f:
        for line in f:
            m = ASIN_RE.search(line)
            if not m or m.group(1) not in asins:
                continue
            d = ast.literal_eval(line)
            rows.append({
                "asin": d["asin"],
                "title": html.unescape(d.get("title", "")) or None,
                "brand": d.get("brand") or None,
                "price": d.get("price"),
                "description": html.unescape(d.get("description", "")) or None,
                "categories": [" > ".join(c) for c in d.get("categories", [])],
            })
    return pd.DataFrame(rows, columns=["asin", "title", "brand", "price", "description", "categories"])

## 1. Bitext Customer Support

Single `train` split with the columns `flags`, `instruction` (customer question), `category`, `intent` and `response` (reference answer). Placeholders such as `{{Order Number}}` are part of the data.

In [ ]:
from datasets import load_dataset

bitext_path = BITEXT_DIR / "bitext_customer_support.jsonl"
if bitext_path.exists():
    bitext = read_jsonl(bitext_path)
    print(f"skip (exists): {bitext_path.name}")
else:
    bitext = load_dataset(BITEXT_DATASET, split="train").to_pandas()
    write_jsonl(bitext, bitext_path)

print(f"{len(bitext):,} rows, {bitext['category'].nunique()} categories, {bitext['intent'].nunique()} intents")
bitext.head()

In [ ]:
# Rows per category and intent: the basis for stratified sampling of the golden set later.
bitext.groupby(["category", "intent"]).size().rename("rows").to_frame()

## 2. Amazon Q/A + product metadata

Downloads `qa_<Category>.json.gz` and `meta_<Category>.json.gz`, then joins them on `asin`.

In [ ]:
for cat in AMAZON_CATEGORIES:
    download(AMAZON_QA_URL.format(cat=cat), AMAZON_DIR / f"qa_{cat}.json.gz")
    download(AMAZON_META_URL.format(cat=cat), AMAZON_DIR / f"meta_{cat}.json.gz")

In [ ]:
QA_COLUMNS = ["asin", "questionType", "answerType", "question", "answer", "answerTime", "unixTime"]

summary = []
for cat in AMAZON_CATEGORIES:
    out_path = AMAZON_DIR / f"amazon_qa_{cat}.jsonl"
    if out_path.exists():
        joined = read_jsonl(out_path)
        print(f"skip (exists): {out_path.name}")
    else:
        qa = pd.DataFrame(iter_py_literals(AMAZON_DIR / f"qa_{cat}.json.gz")).reindex(columns=QA_COLUMNS)
        meta = load_meta_for(AMAZON_DIR / f"meta_{cat}.json.gz", set(qa["asin"]))
        joined = qa.merge(meta, on="asin", how="left")
        joined.insert(0, "category", cat)
        write_jsonl(joined, out_path)

    summary.append({
        "category": cat,
        "questions": len(joined),
        "products": joined["asin"].nunique(),
        "with_title": joined["title"].notna().mean(),
        "with_description": joined["description"].notna().mean(),
    })

pd.DataFrame(summary).style.format({"with_title": "{:.1%}", "with_description": "{:.1%}"})

`with_title` / `with_description` show how many questions have product context. Questions without title or description cannot be answered from context alone and should stay out of the golden set.

In [ ]:
amazon = pd.concat(
    [read_jsonl(AMAZON_DIR / f"amazon_qa_{cat}.jsonl") for cat in AMAZON_CATEGORIES],
    ignore_index=True,
)
amazon_with_context = amazon.dropna(subset=["title", "description"])
print(f"{len(amazon):,} questions total, {len(amazon_with_context):,} with title + description")
amazon_with_context[["category", "questionType", "question", "answer", "title", "description"]].sample(5, random_state=42)

## Output

Files in `src/data/raw/`:

In [ ]:
for p in sorted(RAW_DIR.rglob("*")):
    if p.is_file():
        print(f"{p.relative_to(RAW_DIR)!s:<45} {p.stat().st_size / 1e6:8.1f} MB")